# Retail Footprint Expansion Model: walkthrough

This notebook walks through the whole method, one step at a time, using the functions in `src/footprint/`.

> **Data disclaimer:** all retailer, sales and store data here is **simulated**. It is not real company data and exists only to explain the method. District boundaries (geoBoundaries) and population (Pakistan Population Census 2023) are public.

**Short forms used in this notebook**

| Short form | Meaning |
|---|---|
| PCC | Per Capita Consumption: liters per target person per year |
| PPO | Population Per Outlet: target people for every freezer outlet |
| TP | Throughput: liters sold per freezer per week (one outlet means one freezer) |
| NI | New Induction: a retailer that received a freezer during the year |
| LSM 5+ | Living Standards Measure 5 and above: people who can afford ice cream regularly |
| GOLY / CAGR | Growth Over Last Year / Compound Annual Growth Rate |

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # run this notebook from the notebooks/ folder

import pandas as pd
from footprint import census, config, geo, model, pipeline, rollout

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

## Step 1. Load the data

* **Public:** official district boundaries and 2023 Census population.
* **Simulated:** retailers (with GPS coordinates and the district name typed by the sales team), and the retail census used in Phase 2.

In [2]:
districts, population, attrs, retailers, census_stores = pipeline.load_inputs()
print(f"{len(districts)} districts, {len(retailers):,} retailers, {len(census_stores):,} census stores")
retailers.head()

115 districts, 41,544 retailers, 189,795 census stores


,retailer_id,region,distributor_id,channel,district_typed,latitude,longitude,induction_year,status,volume_2022,volume_2023,volume_2024
0,R000001,Karachi Region,DST-KAR-01,Out of Home,khuzdar,27.26,66.64,2023,Active,0.00,392.50,628.70
1,R000002,Karachi Region,DST-KAR-01,Traditional Trade,Khuzdar,27.68,67.40,2021 or earlier,Active,734.90,713.70,693.10
2,R000003,Karachi Region,DST-KAR-01,Traditional Trade,Khuzdaar,27.22,66.41,2021 or earlier,Active,"1,948.00","1,891.80","1,837.20"
3,R000004,Karachi Region,DST-KAR-01,Traditional Trade,Khuzdar District,27.71,67.42,2024,Active,0.00,0.00,710.80
4,R000005,Karachi Region,DST-KAR-01,Traditional Trade,KHUZDAR,27.29,66.44,2021 or earlier,Active,"1,007.40",978.30,950.10


## Step 2. The data problem: one district, many spellings

Grouping sales by the typed district name would create hundreds of fake districts. Here is just one district:

In [3]:
lahore_spellings = retailers[retailers["district_typed"].fillna("").str.lower().str.contains("lah|lhr")]
lahore_spellings["district_typed"].value_counts().head(12)

,count
district_typed,
Lahore,5457
lahore,1111
Lahore City,490
LHR,484
Lahore Cantt,466
LAHORE,457
LAHORE,444
Lahore District,442
Lahre,179


## Step 3. The fix: map every retailer to its district using GPS coordinates

Each retailer's latitude and longitude is checked against the official district boundaries (a point in polygon test).
The original version (`legacy/main.py`) looped over every retailer and every district. The rebuilt version uses a GeoPandas spatial join, and also repairs swapped coordinates and flags missing ones.

![Retailers mapped to districts](../outputs/figures/01_retailers_mapped_to_districts.png)

In [4]:
retailers_mapped = geo.map_points_to_districts(retailers, districts)
retailers_mapped["mapping_status"].value_counts()

,count
mapping_status,
Mapped,41452
Invalid (outside Pakistan or missing),92


In [5]:
print("Distinct typed names:", retailers_mapped["district_typed"].fillna("").str.strip().nunique())
print("Official districts after mapping:", retailers_mapped["district_mapped"].nunique())

Distinct typed names: 1101
Official districts after mapping: 114


## Step 4. District metrics (Phase 1)

For every district:

* **Target population** = Population x LSM 5+ share
* **PCC** = Volume / Target population
* **PPO** = Target population / Outlets
* **TP** = Volume / Outlets / 52 weeks
* **NI TP** = the same, for retailers that received a freezer in 2024

In [6]:
metrics = model.district_metrics(retailers_mapped, attrs, population)
cols = ["district", "region", "nature_of_district", "retailers_2024", "volume_2024", "target_population_2024",
        "pcc_2024", "ppo_2024", "tp_2024", "ni_tp_2024", "volume_cagr", "pcc_layer", "ppo_layer"]
metrics.sort_values("volume_2024", ascending=False)[cols].head(10)

,district,region,nature_of_district,retailers_2024,volume_2024,target_population_2024,pcc_2024,ppo_2024,tp_2024,ni_tp_2024,volume_cagr,pcc_layer,ppo_layer
53,Lahore,Lahore Region,Stronghold,10222,"10,464,261.90","9,476,962.00",1.10,927.11,19.69,9.78,0.02,Layer 02,Layer 05
38,Karachi,Karachi Region,Stronghold,8049,"8,864,528.00","14,681,988.00",0.60,"1,824.08",21.18,14.58,0.02,Layer 03,Layer 04
22,Gujranwala,Lahore Region,Stronghold,2722,"3,231,103.30","2,926,152.00",1.10,"1,075.00",22.83,14.63,0.02,Layer 02,Layer 05
20,Faisalabad,Lahore Region,Stronghold,2113,"2,648,083.10","3,790,953.00",0.70,"1,794.11",24.10,15.17,0.04,Layer 03,Layer 04
28,Hyderabad,Karachi Region,Stronghold,2003,"1,867,284.00","1,649,909.00",1.13,823.72,17.93,10.69,0.03,Layer 02,Layer 05
98,Sialkot,Lahore Region,Stronghold,1093,"1,209,694.40","1,585,465.00",0.76,"1,450.56",21.28,13.89,0.05,Layer 03,Layer 05
91,Rawalpindi,Islamabad Region,Stronghold,1148,"1,033,311.30","3,548,424.00",0.29,"3,090.96",17.31,8.59,0.00,Layer 04,Layer 04
83,Peshawar,Peshawar Region,Stronghold,748,"785,425.10","1,822,363.00",0.43,"2,436.31",20.19,6.78,0.03,Layer 04,Layer 04
69,Multan,Multan Region,Stronghold,814,"728,019.20","2,320,340.00",0.31,"2,850.54",17.20,7.10,0.04,Layer 04,Layer 04
23,Gujrat,Lahore Region,Stronghold,655,"649,361.10","1,281,869.00",0.51,"1,957.05",19.07,14.35,0.02,Layer 03,Layer 04


## Step 5. How many freezers to add?

Every new freezer adds volume (first fill plus weekly refills at the NI TP), but it also spreads demand across more freezers, so **TP per freezer falls**.
We keep adding freezers until TP reaches the **Ideal TP (15 liters per week)** or PPO reaches the **Ideal PPO (750)**, whichever comes first.

New Volume(n) = Current Volume + n x (52 x New outlet TP + 80 liters first fill)

The original Excel model used Goal Seek for each district. Here the equation is solved directly.

![TP curve](../outputs/figures/05_tp_curve_example.png)

In [7]:
recommended = model.recommend(metrics)
recommended["limiting_factor"].value_counts()

,count
limiting_factor,
TP already at or below ideal,68
Stopped by TP reaching ideal,31
Stopped by PPO reaching ideal,15
No current outlets,1


In [8]:
# One district, step by step
r = recommended.set_index("district").loc["Peshawar"]
print(f"Outlets today: {r.retailers_2024:,.0f}   TP today: {r.tp_2024:.1f}   PPO today: {r.ppo_2024:,.0f}")
print(f"TP assumed per new outlet (NI TP): {r.new_outlet_tp:.1f}")
print(f"Recommended new outlets: {r.recommended_new_outlets:,}  ({r.limiting_factor})")
print(f"After expansion: TP {r.new_tp:.2f}, PPO {r.new_ppo:,.0f}, volume +{r.incremental_volume:,.0f} liters per year")

Outlets today: 748   TP today: 20.2   PPO today: 2,436
TP assumed per new outlet (NI TP): 6.8
Recommended new outlets: 581  (Stopped by TP reaching ideal)
After expansion: TP 15.00, PPO 1,371, volume +251,390 liters per year


## Step 6. Prioritize: two quadrant views and the expansion waves

![PCC against PPO](../outputs/figures/03_quadrant_pcc_vs_ppo.png)

![TP against PPO](../outputs/figures/04_quadrant_tp_vs_ppo.png)

Expansion order from the strategy: strongholds first, then their neighbouring districts, then high population districts, then the rest.

In [9]:
prioritized = model.prioritize(recommended)
prioritized[prioritized["shortlisted"]].sort_values("priority_rank")[
    ["priority_rank", "district", "nature_of_district", "phase", "quadrant_tp_ppo", "recommended_new_outlets"]
].head(15)

,priority_rank,district,nature_of_district,phase,quadrant_tp_ppo,recommended_new_outlets
38,1.00,Karachi,Stronghold,Phase 1,"High TP, High PPO",11526
20,2.00,Faisalabad,Stronghold,Phase 1,"High TP, High PPO",2941
53,3.00,Lahore,Stronghold,Phase 1,"High TP, High PPO",2413
22,4.00,Gujranwala,Stronghold,Phase 1,"High TP, High PPO",1179
23,5.00,Gujrat,Stronghold,Phase 1,"High TP, High PPO",1054
98,6.00,Sialkot,Stronghold,Phase 1,"High TP, High PPO",1020
28,7.00,Hyderabad,Stronghold,Phase 1,"High TP, High PPO",196
83,8.00,Peshawar,Stronghold,Phase 3,"High TP, High PPO",581
91,9.00,Rawalpindi,Stronghold,Phase 3,"High TP, High PPO",544
96,10.00,Sheikhpura,Stronghold,Phase 3,"High TP, High PPO",387


## Step 7. Phase 2: do enough real shops exist?

A third party retail census lists stores with GPS coordinates. A store is **high potential** only if it sells all the main snacking competitors (Coke or Pepsi, Dairy Milk, Lays, Sooper), is in a target channel, and has high turnover.
Stores that are already customers are removed (matched by location), and only 80 percent of the rest are counted (conversion rate).

![Census funnel](../outputs/figures/07_census_funnel.png)

In [10]:
stores = census.run(census_stores, retailers_mapped, districts)
supply = census.district_supply(stores)
checked = census.reality_check(prioritized, supply)
checked[checked["shortlisted"]]["constraint"].value_counts()

,count
constraint,
Supply constrained (not enough shops),28
Demand constrained (enough shops),16


**Final recommendation = the smaller of Phase 1 demand and Phase 2 usable supply.**

![Demand against supply](../outputs/figures/06_demand_vs_supply.png)

In [11]:
checked[checked["shortlisted"]].sort_values("priority_rank")[
    ["district", "recommended_new_outlets", "usable_supply", "final_new_outlets", "constraint"]
].head(15)

,district,recommended_new_outlets,usable_supply,final_new_outlets,constraint
38,Karachi,11526,1906,1906,Supply constrained (not enough shops)
20,Faisalabad,2941,400,400,Supply constrained (not enough shops)
53,Lahore,2413,1336,1336,Supply constrained (not enough shops)
22,Gujranwala,1179,464,464,Supply constrained (not enough shops)
23,Gujrat,1054,60,60,Supply constrained (not enough shops)
98,Sialkot,1020,169,169,Supply constrained (not enough shops)
28,Hyderabad,196,308,196,Demand constrained (enough shops)
83,Peshawar,581,239,239,Supply constrained (not enough shops)
91,Rawalpindi,544,531,531,Supply constrained (not enough shops)
96,Sheikhpura,387,184,184,Supply constrained (not enough shops)


## Step 8. Turn it into a plan

* **Budget scenarios:** if only a fixed number of freezers is funded, share them by each district's contribution.
* **Phasing:** split each district across 2025, 2026 and 2027 (300, 300 and 250 out of every 850).
* **Sales target list:** the exact stores to visit, highest turnover first.

In [12]:
plan = rollout.build_plan(checked)
budget_cols = [f"budget_{b}" for b in config.BUDGET_SCENARIOS]
year_cols = [f"rollout_{y}" for y in config.PHASING]
plan[["final_new_outlets", *budget_cols, *year_cols]].sum().to_frame("total freezers")

,total freezers
final_new_outlets,6765
budget_4000,4000
budget_6000,6000
rollout_2025,2398
rollout_2026,2379
rollout_2027,1988


In [13]:
targets = rollout.sales_target_list(stores, plan)
targets.head(10)

,priority_rank,district,rollout_year,store_id,channel,turnover_band,est_monthly_turnover_pkr,latitude,longitude,phase
0,1,Karachi,2025,CS0018614,Pharmacy,High,1968825,24.86,66.90,Phase 1
1,1,Karachi,2025,CS0021592,Grocery,High,1922006,25.11,67.24,Phase 1
2,1,Karachi,2025,CS0023459,Bakery,High,1907674,24.79,67.16,Phase 1
3,1,Karachi,2025,CS0036701,Pharmacy,High,1853363,25.11,67.13,Phase 1
4,1,Karachi,2025,CS0046635,Bakery,High,1818241,25.18,67.27,Phase 1
5,1,Karachi,2025,CS0018699,Supermarket,High,1756299,25.07,67.23,Phase 1
6,1,Karachi,2025,CS0023772,Grocery,High,1736663,25.08,67.29,Phase 1
7,1,Karachi,2025,CS0048832,Supermarket,High,1734387,24.87,66.98,Phase 1
8,1,Karachi,2025,CS0034283,Grocery,High,1728418,25.21,67.31,Phase 1
9,1,Karachi,2025,CS0018741,Bakery,High,1679572,25.15,67.19,Phase 1


## Step 9. What if? (the model is driven by settings)

Every setting lives in `src/footprint/config.py` and on the Inputs sheet of the Excel model. For example: use the district average TP for new outlets instead of NI TP.

In [14]:
alt = model.recommend(metrics, basis="district_tp")
print(f"New outlets with NI TP (original method): {recommended['recommended_new_outlets'].sum():,}")
print(f"New outlets with district average TP:     {alt['recommended_new_outlets'].sum():,}")

New outlets with NI TP (original method): 26,899
New outlets with district average TP:     49,039


## Excel and VBA versions

* `excel/Retail_Footprint_Expansion_Model.xlsx`: the same model as live Excel formulas, for colleagues who do not code. Change the Inputs sheet and everything recalculates.
* `vba/FootprintModel.bas`: a one click macro that replaces manual Goal Seek, adding outlets one at a time and checking its answer against the formulas.

All three versions (Python, Excel formulas, VBA) give the same answer for every district. This is checked by `tests/test_excel_matches_python.py` and `scripts/check_vba_with_libreoffice.py`.